In [28]:
import seaborn
from scipy.special import softmax, log_softmax
from numpy.lib.stride_tricks import sliding_window_view
import scipy.stats as stats
import pickle
import json
from pathlib import Path
import warnings
import torch
import re
from collections import defaultdict
import matplotlib.pyplot as plt
import numpy as np
from seaborn import kdeplot
from tqdm.auto import tqdm
from scipy.stats import gmean


def meanprob(answer_probabilities):
    return gmean([next(iter(value.values())) for value in answer_probabilities])


def meanprob_dropout(answer_probabilities):
    return gmean([meanprob(sample) for sample in answer_probabilities])


def is_same_literal_regex(x, y):
    # Remove anything that isn't a letter or a number
    clean_x = re.sub(r'[^a-zA-Z0-9]', '', x).upper()
    clean_y = re.sub(r'[^a-zA-Z0-9]', '', y).upper()
    
    return clean_x == clean_y

In [29]:
sample_filepaths = {}
sample_filepaths["coinflip"] = Path("./trajectories/test-422-pre-llama-coinflip-10_llama_regular_bigbench_movie_zero_0_250_type2_conf_coinflip_mod-none")

In [30]:
confidences = defaultdict(lambda: defaultdict(list))
nb_steps = defaultdict(list)
nb_jackknifes = defaultdict(list)
generated_text = defaultdict(list)
for dataset in sample_filepaths:
    print(f"Loading {dataset}")
    trajectories = []
    sample_files = list(sample_filepaths[dataset].glob("traj_*.json"))
    for sample_file in tqdm(sample_files):
        with open(sample_file, "r") as file:
            trajectories += [json.load(file)]
    
    for trajectory in tqdm(trajectories):
        traj_conf = trajectory['trajectory']['confidence_score']
        if traj_conf is not None:
            generated_text[dataset] += [trajectory['trajectory']['generated_text']]
            nb_steps[dataset] += [len(trajectory['trajectory']['cot_steps'])]
            confidences[dataset]['accuracy'] += [is_same_literal_regex(trajectory['trajectory']['final_answer'], 
                                                                       trajectory['trajectory']['ground_truth'])]

            if traj_conf["vanilla_confidences"] is not None:
                confidences[dataset]['vanilla_meanprob'] += [meanprob(traj_conf['vanilla_confidences']['answer_probabilities'])]
                confidences[dataset]['vanilla_meanent'] += [np.exp(-meanprob(traj_conf['vanilla_confidences']['answer_entropy']))]
                confidences[dataset]['vanilla_indirect1'] += [traj_conf['vanilla_confidences']['indirect_ptrue1_probabilities'][0]['True']]
                confidences[dataset]['vanilla_indirect2'] += [traj_conf['vanilla_confidences']['indirect_ptrue2_probabilities'][0]['Yes']]
                confidences[dataset]['vanilla_indirect3'] += [traj_conf['vanilla_confidences']['indirect_ptrue3_probabilities'][0]['Yes']]
                confidences[dataset]['vanilla_verbal'] += [traj_conf['vanilla_confidences']['verbconf_probabilities'][0]]

            if traj_conf["dropout_confidences"] is not None:
                confidences[dataset]['dropout_meanprob'] += [meanprob_dropout(traj_conf['dropout_confidences']['answer_probabilities'])]
                confidences[dataset]['dropout_meanent'] += [np.exp(-meanprob_dropout(traj_conf['dropout_confidences']['answer_entropy']))]
                confidences[dataset]['dropout_indirect1'] += [traj_conf['dropout_confidences']['indirect_ptrue1_probabilities'][0]['True']]
                confidences[dataset]['dropout_indirect2'] += [traj_conf['dropout_confidences']['indirect_ptrue2_probabilities'][0]['Yes']]
                confidences[dataset]['dropout_indirect3'] += [traj_conf['dropout_confidences']['indirect_ptrue3_probabilities'][0]['Yes']]
                confidences[dataset]['dropout_verbal'] += [traj_conf['dropout_confidences']['verbconf_probabilities'][0]]

            if traj_conf["jackknife_confidences"] is not None:
                confidences[dataset]['jackknife_meanprob'] += [[meanprob(confidence) for confidence in traj_conf['jackknife_confidences']['answer_probabilities']]]
                confidences[dataset]['jackknife_meanent'] += [[np.exp(-meanprob(confidence)) for confidence in traj_conf['jackknife_confidences']['answer_entropy']]]
                confidences[dataset]['jackknife_indirect1'] += [[confidence['True'] for confidence in traj_conf['jackknife_confidences']['indirect_ptrue1_probabilities']]]
                confidences[dataset]['jackknife_indirect2'] += [[confidence['Yes'] for confidence in traj_conf['jackknife_confidences']['indirect_ptrue2_probabilities']]]
                confidences[dataset]['jackknife_indirect3'] += [[confidence['Yes'] for confidence in traj_conf['jackknife_confidences']['indirect_ptrue3_probabilities']]]
                confidences[dataset]['jackknife_verbal'] += [traj_conf['jackknife_confidences']['verbconf_probabilities']]

    
    nb_steps[dataset] = torch.LongTensor(nb_steps[dataset])
    nb_jackknifes[dataset] = torch.ceil(torch.sqrt(nb_steps[dataset]))
    for key in confidences[dataset]:
        confidences[dataset][key] = torch.FloatTensor(confidences[dataset][key])

Loading coinflip


100%|████████████████████████████████████████████████████████████████████████████| 250/250 [00:00<00:00, 310.27it/s]


In [33]:


traj = trajectories[0]

traj['trajectory']['confidence_score']['dropout_confidences'].keys()


dict_keys(['answer_probabilities', 'answer_entropy', 'indirect_ptrue1_probabilities', 'indirect_ptrue2_probabilities', 'indirect_ptrue3_probabilities', 'verbconf_probabilities', 'verbconf_distribution', 'verbconf_top_score', 'verbconf_top_prob', 'step_masks'])

In [22]:
confidences['coinflip'].keys()



dict_keys(['accuracy', 'vanilla_meanprob', 'vanilla_meanent', 'vanilla_indirect1', 'vanilla_indirect2', 'vanilla_indirect3', 'vanilla_verbal'])

In [38]:
for i, traj in enumerate(trajectories):
    step_masks = traj['trajectory']['confidence_score']['dropout_confidences']['step_masks']
    for mask in step_masks:
        if mask[-1] == 0:
            print(i)
            